In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/test_split.csv')

# Multi-label binary columns
for ae in ['Falls', 'Fractures', 'GI_Bleed', 'Stroke']:
    df[ae] = df['labels'].str.contains(ae, na=False).astype(int)
df['No_AE'] = (df['labels'] == 'no').astype(int)

# Sex: 1=Male, 2=Female
df['sex_label'] = df['gender'].map({1: 'Male', 2: 'Female'})

# Age bins
bins = [0, 40, 50, 60, 70, 80, 200]
labels = ['0--40', '41--50', '51--60', '61--70', '71--80', '81+']
df['age_bin'] = pd.cut(df['ageYear'], bins=bins, labels=labels, right=True)

ae_cols = ['Falls', 'Fractures', 'GI_Bleed', 'Stroke', 'No_AE']

def count_row(sub, label):
    r = {'Subgroup': label, 'n': len(sub)}
    for c in ae_cols:
        r[c] = sub[c].sum()
    return r

rows = []

# By Age
for ab in labels:
    rows.append(('Age', count_row(df[df['age_bin'] == ab], ab)))

# By Sex
for s in ['Female', 'Male']:
    rows.append(('Sex', count_row(df[df['sex_label'] == s], s)))

# Female x Age
for ab in labels:
    sub = df[(df['sex_label'] == 'Female') & (df['age_bin'] == ab)]
    rows.append(('Female x Age', count_row(sub, ab)))

# Male x Age
for ab in labels:
    sub = df[(df['sex_label'] == 'Male') & (df['age_bin'] == ab)]
    rows.append(('Male x Age', count_row(sub, ab)))

# Build table
out = []
for strat, r in rows:
    out.append({'Stratification': strat, **r})
result = pd.DataFrame(out)
print(result.to_string(index=False))

In [ ]:
# Generate LaTeX
def fmt(n):
    """Format integer with LaTeX thousands separator."""
    s = f'{int(n):,}'
    return s.replace(',', '{,}')

tex_lines = []
tex_lines.append(r'\begin{table}[htbp]')
tex_lines.append(r'    \centering')
tex_lines.append(r'    \caption{Number of patients with each adverse event per subgroup (MIMIC-IV test set).}')
tex_lines.append(r'    \label{tab:supp_prevalence}')
tex_lines.append(r'    \resizebox{\textwidth}{!}{%')
tex_lines.append(r'    \begin{tabular}{|l|l|r|r|r|r|r|r|}')
tex_lines.append(r'    \hline')
tex_lines.append(r'    \textbf{Stratification} & \textbf{Subgroup} & \textbf{$n$ (total)} & \textbf{Falls} & \textbf{Fractures} & \textbf{GI Bleed} & \textbf{Stroke} & \textbf{No AE} \\\\ \\hline')

strat_map = {
    'Age': ('Age', 6),
    'Sex': ('Sex', 2),
    'Female x Age': (r'\\shortstack[l]{Female\\\\$\\times$ Age}', 6),
    'Male x Age': (r'\\shortstack[l]{Male\\\\$\\times$ Age}', 6),
}

prev_strat = None
for _, row in result.iterrows():
    strat = row['Stratification']
    sub = row['Subgroup']
    vals = ' & '.join([fmt(row[c]) for c in ['n'] + ae_cols])
    if strat != prev_strat:
        label, nrows = strat_map[strat]
        tex_lines.append(f'    \\multirow{{{nrows}}}{{*}}{{{label}}}')
        prev_strat = strat
    tex_lines.append(f'    & {sub:7s} & {vals} \\\\\\\\')
    # Check if this is last row of group
    idx = result.index[result.index == _][0]
    if idx + 1 >= len(result) or result.loc[idx + 1, 'Stratification'] != strat:
        tex_lines[-1] = tex_lines[-1] + ' \\hline'

tex_lines.append(r'    \end{tabular}%')
tex_lines.append(r'    }')
tex_lines.append(r'\end{table}')

print('\n'.join(tex_lines))